In [1]:
import pandas as pd
import os

In [2]:
BASE_PATH ="../Data/processed"
OUTPUT_PATH ="../sql"
os.makedirs(OUTPUT_PATH, exist_ok=True)
print('Folders Ready')

Folders Ready


In [3]:
patients= pd.read_csv(f"{BASE_PATH}/patients_processed.csv")
providers= pd.read_csv(f"{BASE_PATH}/providers_processed.csv")
departments= pd.read_csv(f"{BASE_PATH}/departments_processed.csv")
diagnoses= pd.read_csv(f"{BASE_PATH}/diagnoses_processed.csv")
visits= pd.read_csv(f"{BASE_PATH}/visits_processed.csv")

In [4]:
print(patients.shape)
print(providers.shape)
print(departments.shape)
print(diagnoses.shape)
print(visits.shape)

(5000, 18)
(40, 8)
(8, 2)
(25, 4)
(20000, 27)


In [6]:
def clean_sql_value(value):
    """
    Converts Python values into SQL-safe values.
    """

    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        value = value.replace("\\", "\\\\")
        value = value.replace("'", "''")
        return f"'{value}'"

    return str(value)



In [7]:
# Reproducible SQL generator

def dataframe_to_sql(df, table_name, output_folder):

    output_file = os.path.join(
        output_folder,
        f"{table_name}_insert.sql"
    )

    columns = ", ".join(df.columns)

    with open(output_file, "w", encoding="utf-8") as file:

        file.write("USE healthcare_outpatient_analytics;\n\n")

        file.write(f"DELETE FROM {table_name};\n\n")

        for _, row in df.iterrows():

            values = ", ".join(
                clean_sql_value(v)
                for v in row
            )

            sql = (
                f"INSERT INTO {table_name} "
                f"({columns}) "
                f"VALUES ({values});\n"
            )

            file.write(sql)

    print(f"{table_name} completed.")

In [8]:
# Generate every table
dataframe_to_sql(
    departments,
    "departments",
    OUTPUT_PATH
)
dataframe_to_sql(
    diagnoses,
    "diagnoses",
    OUTPUT_PATH
)
dataframe_to_sql(
    providers,
    "providers",
    OUTPUT_PATH
)
dataframe_to_sql(
    patients,
    "patients",
    OUTPUT_PATH
)
dataframe_to_sql(
    visits,
    "visits",
    OUTPUT_PATH
)


departments completed.
diagnoses completed.
providers completed.
patients completed.
visits completed.


In [10]:
import os
print("SQL Folder:",OUTPUT_PATH)
print()
print("Generated Files:")
for file in os.listdir(OUTPUT_PATH):
    print(file)

SQL Folder: ../sql

Generated Files:
.ipynb_checkpoints
departments_insert.sql
diagnoses_insert.sql
patients_insert.sql
providers_insert.sql
visits_insert.sql
